In [1]:
import pyomo.environ as pe
import pyomo.opt as po
solver = po.SolverFactory('appsi_highs')
model = pe.ConcreteModel()

In [2]:
processing_times=[
  [3,3,5],
  [2,5,6],
  [3,2,5]
]
profit_per_product = [30,35,32]
max_processing_time = [600,800,1200]

amount=3

In [3]:
model.J = pe.RangeSet(1,amount)
model.I = pe.RangeSet(1,amount)
model.p = pe.Param(model.I, initialize=lambda m, i: profit_per_product[i-1])
model.h = pe.Param(model.J,initialize=lambda m, j: max_processing_time[j-1])
model.a = pe.Param(model.I, model.J, initialize=lambda m, i, j: processing_times[i-1][j-1])

In [4]:
model.x=pe.Var(model.I,domain=pe.NonNegativeReals)

In [5]:
obj_expr=sum(model.p[i]*model.x[i] for i in model.I)
model.obj=pe.Objective(sense=pe.maximize, expr=obj_expr)

In [6]:
def con_expr1(model, j):
    return sum(model.a[i,j] * model.x[i] for i in model.I) <= model.h[j] 
model.con1 = pe.Constraint(model.J, rule=con_expr1)
result = solver.solve(model)

In [7]:
model.display()

Model unknown

  Variables:
    x : Size=3, Index=I
        Key : Lower : Value              : Upper : Fixed : Stale : Domain
          1 :     0 :                0.0 :  None : False : False : NonNegativeReals
          2 :     0 :  74.99999999999996 :  None : False : False : NonNegativeReals
          3 :     0 : 150.00000000000006 :  None : False : False : NonNegativeReals

  Objectives:
    obj : Size=1, Index=None, Active=True
        Key  : Active : Value
        None :   True : 7425.0

  Constraints:
    con1 : Size=3
        Key : Lower : Body              : Upper
          1 :  None : 600.0000000000001 :  600.0
          2 :  None : 674.9999999999999 :  800.0
          3 :  None :            1200.0 : 1200.0


In [8]:
for i in model.I:
  print(f"x{i}:",pe.value(model.x[i]))
print("Goal:",pe.value(model.obj))

x1: 0.0
x2: 74.99999999999996
x3: 150.00000000000006
Goal: 7425.0
